In [1]:
import os, re, json, glob, unicodedata, textwrap
from typing import List, Dict
import pandas as pd
from tqdm import tqdm
import chess
import chess.pgn

INPUT_DIR = 'data/annotated_pgn_free'
CLEAN_DIR = 'data/clean'
REWRITTEN_DIR = 'data/rewritten'
FINAL_DIR = 'data/final'
EXISTING_DATASET = 'data/game_knot_final.csv'
CLEAN_OUTPUT = os.path.join(CLEAN_DIR, 'lichess_cleaned.csv')
REWRITTEN_INPUT = os.path.join(REWRITTEN_DIR, 'lichess_rewritten.csv')
FINAL_OUTPUT = os.path.join(FINAL_DIR, 'annotated_positions_final.csv')
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(REWRITTEN_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

MIN_WORDS = 6
MAX_WORDS = 10000
invalid_fen_count = 0

# 2. PGN parsing with comments

Extract comments by move and generate `fen` and `board_matrix` (8x8) per position.

In [ ]:
def fen_to_board_matrix(fen: str) -> List[List[str]]:
    board_part = fen.split()[0]
    rows = []
    for row_str in board_part.split('/'):
        row = []
        for ch in row_str:
            if ch.isdigit():
                row.extend(['.'] * int(ch))
            else:
                row.append(ch)
        rows.append(row)
    if len(rows) != 8 or any(len(r) != 8 for r in rows):
        raise ValueError(f'Invalid FEN: {fen}')
    return rows


def clean_comment(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def iter_game_comments(game: chess.pgn.Game, game_id: str, source: str) -> List[Dict]:
    global invalid_fen_count
    rows = []
    board = game.board()
    # Root comment (before any move)
    if game.comment:
        comment = clean_comment(game.comment)
        if comment:
            fen = board.fen()
            try:
                board_matrix = fen_to_board_matrix(fen)
            except ValueError:
                invalid_fen_count += 1 
                print(f"Invalid FEN {fen} {invalid_fen_count}")
                board_matrix = None
            if board_matrix is not None:
                rows.append({
                    'game_id': game_id,
                    'fen': fen,
                    'board_matrix': board_matrix,
                    'annotation': comment,
                    'source': source,
                })
    for node in game.mainline():
        if node.comment:
            comment = clean_comment(node.comment)
            if comment:
                fen = board.fen()
                try:
                    board_matrix = fen_to_board_matrix(fen)
                except ValueError:
                    invalid_fen_count += 1
                    print(f"Invalid FEN {fen} {invalid_fen_count}")
                    board_matrix = None
                if board_matrix is not None:
                    rows.append({
                        'game_id': game_id,
                        'fen': fen,
                        'board_matrix': board_matrix,
                        'annotation': comment,
                        'source': source,
                    })
        # Move to the next position
        if node.move is not None:
            board.push(node.move)
    return rows


def parse_pgn_file(path: str, base_id: int) -> List[Dict]:
    rows = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        idx = 0
        while True:
            game = chess.pgn.read_game(f)
            if game is None:
                break
            gid = f'{base_id}_{idx}'
            rows.extend(iter_game_comments(game, gid, os.path.relpath(path)))
            idx += 1
    return rows


def load_all_pgns(input_dir: str) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(input_dir, '**', '*.pgn'), recursive=True))
    all_rows = []
    base = 0
    for path in tqdm(files, desc='PGN files'):
        all_rows.extend(parse_pgn_file(path, base))
        base += 1000000
    df = pd.DataFrame(all_rows, columns=['game_id', 'fen', 'board_matrix', 'annotation', 'source'])
    df['board_matrix'] = df['board_matrix'].apply(json.dumps)
    return df


df_raw = load_all_pgns(INPUT_DIR)
df_raw.head()

# 3. Basic deduplication and normalization

Remove empty entries and duplicates by (fen, annotation).

In [ ]:
df_raw = df_raw.dropna(subset=['annotation', 'fen']).copy()
df_raw = df_raw[df_raw['annotation'].str.len() > 0]
df_raw = df_raw.drop_duplicates(subset=['fen', 'annotation'])
df_raw.reset_index(drop=True, inplace=True)
df_raw.head()

# 4. Language detection (FastText lid.176)

Keep only English comments.

In [ ]:
import fasttext

LANG_MODEL = 'lid.176.bin'
ft_model = fasttext.load_model(LANG_MODEL)

def detect_ft(text: str) -> str:
    try:
        label = ft_model.predict(text)[0][0]
        return label.replace('__label__', '')
    except Exception:
        return 'unknown'

df_raw['language'] = df_raw['annotation'].apply(detect_ft)
df_en = df_raw[df_raw['language'] == 'en'].copy()
df_en.head()

# 5. Notation expansion and cleaning

Replicates the `cleaning.ipynb` flow: piece expansion, ASCII normalization, and word counting.

In [ ]:
piece_dict = {'K': 'king', 'Q': 'queen', 'R': 'rook', 'B': 'bishop', 'N': 'knight'}

def expand_annotation(text: str) -> str:
    if not isinstance(text, str):
        return ''
    t = text
    t = t.replace('O-O-O', 'queenside castle').replace('O-O', 'kingside castle')
    t = re.sub(r'([KQRBN])x([a-h][1-8])', lambda m: piece_dict[m.group(1)] + ' takes ' + m.group(2), t)
    t = re.sub(r'([KQRBN])([a-h][1-8])', lambda m: piece_dict[m.group(1)] + ' to ' + m.group(2), t)
    t = re.sub(r'([a-h])x([a-h][1-8])=([QRBN])', lambda m: 'pawn takes ' + m.group(2) + ' promotes to ' + piece_dict[m.group(3)], t)
    t = re.sub(r'([a-h])([1-8])=([QRBN])', lambda m: 'pawn to ' + m.group(1) + m.group(2) + ' promotes to ' + piece_dict[m.group(3)], t)
    t = re.sub(r'([a-h])x([a-h][1-8])', r'pawn takes \2', t)
    t = t.lower()
    t = unicodedata.normalize('NFKD', t).encode('ascii', errors='ignore').decode('utf-8')
    t = re.sub(r'[^a-z0-9\s.,]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df_en['annotation_expanded'] = df_en['annotation'].apply(expand_annotation)
df_en['annotation_word_count'] = df_en['annotation_expanded'].apply(lambda x: len(x.split()))
df_clean = df_en[(df_en['annotation_word_count'] >= MIN_WORDS) & (df_en['annotation_word_count'] <= MAX_WORDS)].copy()
df_clean.reset_index(drop=True, inplace=True)
df_clean.head()

# 6. Save clean dataset for rewriting

This CSV will be the input for the rewriting notebook.

In [ ]:
df_clean.to_csv(CLEAN_OUTPUT, index=False, encoding='utf-8')
print(f'Clean dataset saved to: {CLEAN_OUTPUT}')
print(f'Rows: {len(df_clean)}')

# 7. PAUSE: run rewriting notebook

Before continuing, open and run all cells in `pgn_rewriting.ipynb` to generate `lichess_rewritten.csv`. Then return and continue with the next cell.

In [ ]:
raise SystemExit('Run pgn_rewriting.ipynb first, then resume here.')

# 8. Load rewritten comments and clean

Remove empty or artifacted descriptions and keep the key columns.

In [13]:
import pandas as pd
df_rew = pd.read_csv(REWRITTEN_INPUT, encoding='utf-8')
df_rew.head()

,game_id,fen,board_matrix,annotation,source,language,annotation_expanded,annotation_word_count,rewritten
0,0_39,r1bqkbnr/pppp1ppp/2n5/4p3/3PP3/5N2/PPP2PPP/RNB...,"[[""r"", ""."", ""b"", ""q"", ""k"", ""b"", ""n"", ""r""], [""p...","In this chapter we will look at the main line,...",data\annotated_pgn_free\lichess_studies\game_s...,en,"in this chapter we will look at the main line,...",18,"in this chapter we will look at the main line,..."
1,0_39,r1bqkbnr/pppp1ppp/2n5/8/3pP3/5N2/PPP2PPP/RNBQK...,"[[""r"", ""."", ""b"", ""q"", ""k"", ""b"", ""n"", ""r""], [""p...",Here there are a couple ways black could respo...,data\annotated_pgn_free\lichess_studies\game_s...,en,here there are a couple ways black could respo...,19,"Description:\nBlack has a knight on f6, sugges..."
2,0_39,r1bqkbnr/pppp1ppp/2n5/8/3NP3/8/PPP2PPP/RNBQKB1...,"[[""r"", ""."", ""b"", ""q"", ""k"", ""b"", ""n"", ""r""], [""p...",Some other ways black could play :,data\annotated_pgn_free\lichess_studies\game_s...,en,some other ways black could play,6,some other ways black could play
3,0_39,r1bqkb1r/p1p2ppp/5n2/3p4/8/3B4/PPP2PPP/RNBQ1RK...,"[[""r"", ""."", ""b"", ""q"", ""k"", ""b"", ""."", ""r""], [""p...",Bc5 is also very popular.,data\annotated_pgn_free\lichess_studies\game_s...,en,bishop to c5 is also very popular.,7,bishop to c5 is also very popular.
4,0_39,r1bq1rk1/p1p1bppp/5n2/3p4/8/3B4/PPP2PPP/RNBQR1...,"[[""r"", ""."", ""b"", ""q"", ""."", ""r"", ""k"", "".""], [""p...","h3 is to prevent Ng4, and Bc5 ideas. [%csl Rg4...",data\annotated_pgn_free\lichess_studies\game_s...,en,"h3 is to prevent knight to g4, and bishop to c...",19,"h3 is to prevent knight to g4, and bishop to c..."


In [3]:
df_rew.shape

(144645, 9)

In [ ]:
for i in range(25):
    print(df_rew.rewritten.iloc[i])
    print('-'*100)

in this chapter we will look at the main line, or most common moves from this point, on.
----------------------------------------------------------------------------------------------------
Description:
Black has a knight on f6, suggesting potential activity or a response to white's last move.
----------------------------------------------------------------------------------------------------
some other ways black could play
----------------------------------------------------------------------------------------------------
bishop to c5 is also very popular.
----------------------------------------------------------------------------------------------------
h3 is to prevent knight to g4, and bishop to c5 ideas. csl rook to g4cal rook to h3g4
----------------------------------------------------------------------------------------------------
white plans to play queen to e2 and complete their development, this is where the line stops.go to the next chapter to see some of the traps.
-----

: 

In [15]:
count = 0
for text in df_rew['rewritten']:
    if re.search(r'^(\d+\.|)$', str(text).strip()):
        count += 1
print(count)

3459


In [ ]:
# Drop rows where 'rewritten' is exactly a number followed by a dot, or only a dot
mask1 = df_rew['rewritten'].apply(lambda x: bool(re.search(r'^(\d+\.|)$', str(x).strip())))

# Drop rows where 'rewritten' does not contain at least one English word
mask2 = ~df_rew['rewritten'].apply(lambda x: any(re.match(r'^[a-zA-Z]+$', word.strip()) for word in str(x).split()))

# Combine masks: drop rows satisfying either condition
mask_to_drop = mask1 | mask2

df_rew = df_rew[~mask_to_drop].copy()
print(f'Rows removed: {mask_to_drop.sum()}')
print(f'Rows remaining: {len(df_rew)}')

Filas eliminadas: 36853
Filas restantes: 107792


In [ ]:
df_rew = pd.read_csv(REWRITTEN_INPUT)
if 'rewritten' not in df_rew.columns:
    raise RuntimeError('The rewritten CSV must contain the rewritten column')
df_rew = df_rew[df_rew['rewritten'].astype(str).str.strip().ne('-')]
df_rew = df_rew[~df_rew['rewritten'].str.contains('You rewrite|\n', na=False)]
df_rew = df_rew.dropna(subset=['fen', 'board_matrix', 'rewritten'])
df_rew.head()

,game_id,fen,board_matrix,annotation,source,language,annotation_expanded,annotation_word_count,rewritten
0,0_39,r1bqkbnr/pppp1ppp/2n5/4p3/3PP3/5N2/PPP2PPP/RNB...,"[[""r"", ""."", ""b"", ""q"", ""k"", ""b"", ""n"", ""r""], [""p...","In this chapter we will look at the main line,...",data\annotated_pgn_free\lichess_studies\game_s...,en,"in this chapter we will look at the main line,...",18,"in this chapter we will look at the main line,..."
2,0_39,r1bqkbnr/pppp1ppp/2n5/8/3NP3/8/PPP2PPP/RNBQKB1...,"[[""r"", ""."", ""b"", ""q"", ""k"", ""b"", ""n"", ""r""], [""p...",Some other ways black could play :,data\annotated_pgn_free\lichess_studies\game_s...,en,some other ways black could play,6,some other ways black could play
3,0_39,r1bqkb1r/p1p2ppp/5n2/3p4/8/3B4/PPP2PPP/RNBQ1RK...,"[[""r"", ""."", ""b"", ""q"", ""k"", ""b"", ""."", ""r""], [""p...",Bc5 is also very popular.,data\annotated_pgn_free\lichess_studies\game_s...,en,bishop to c5 is also very popular.,7,bishop to c5 is also very popular.
4,0_39,r1bq1rk1/p1p1bppp/5n2/3p4/8/3B4/PPP2PPP/RNBQR1...,"[[""r"", ""."", ""b"", ""q"", ""."", ""r"", ""k"", "".""], [""p...","h3 is to prevent Ng4, and Bc5 ideas. [%csl Rg4...",data\annotated_pgn_free\lichess_studies\game_s...,en,"h3 is to prevent knight to g4, and bishop to c...",19,"h3 is to prevent knight to g4, and bishop to c..."
5,0_39,1rbq1rk1/p1p1bppp/5n2/3p4/8/3B3P/PPP2PP1/RNBQR...,"[[""."", ""r"", ""b"", ""q"", ""."", ""r"", ""k"", "".""], [""p...","white according to theory needs to play Bb2, N...",data\annotated_pgn_free\lichess_studies\game_s...,en,white according to theory needs to play bishop...,24,.


# 9. Merge with existing dataset and save final output

Merge the new rewritten dataset with `data/game_knot_final.csv` and save the final CSV.

In [ ]:
# Load previous dataset if it exists
if os.path.exists(EXISTING_DATASET):
    df_prev = pd.read_csv(EXISTING_DATASET)
else:
    df_prev = pd.DataFrame(columns=['game_id', 'fen', 'board_matrix', 'annotation', 'rewritten'])

# Align key columns
for col in ['rewritten']:
    if col not in df_prev.columns:
        df_prev[col] = pd.NA

df_prev = df_prev[['game_id', 'fen', 'board_matrix', 'rewritten']].copy()
df_rew = df_rew[['game_id', 'fen', 'board_matrix', 'rewritten']].copy()

df_final = pd.concat([df_prev, df_rew], ignore_index=True)
df_final = df_final.drop_duplicates(subset=['fen', 'rewritten'])
df_final.to_csv(FINAL_OUTPUT, index=False, encoding='utf-8')
print(f'Final dataset saved to: {FINAL_OUTPUT}')
print(f'Total rows: {len(df_final)}')